<a href="https://colab.research.google.com/github/hemanthvnp/Linear-Discriminant-Analysis/blob/main/notebooks/colab_quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MvDA on Colab

Run Multi-view Discriminant Analysis on Colab, reading the **ColorFERET** images directly from your Google Drive (no manual download).

If you upload a zip of the repo instead of using GitHub, skip cell 1 and `cd` into the unzipped folder.

## 1. Get the code

In [7]:
!git clone https://github.com/hemanthvnp/Linear-Discriminant-Analysis.git mvda
%cd mvda
!pip -q install -r requirements.txt

Cloning into 'mvda'...
remote: Enumerating objects: 63, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 63 (delta 14), reused 58 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (63/63), 41.79 KiB | 6.97 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/mvda/mvda


## 2. Mount Google Drive

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Locate the ColorFERET folder and confirm it has images

If the image count is 0, your Drive copy only has metadata and you'll need the real `.ppm`/`.ppm.bz2` images.

In [9]:
!find /content/drive/MyDrive -maxdepth 4 -iname '*colorferet*' -type d 2>/dev/null | head
FERET_ROOT = '/content/drive/MyDrive/colorferet'  # <-- set to the path printed above
!echo 'image files found:'; find "$FERET_ROOT" -type f \( -iname '*.ppm' -o -iname '*.ppm.bz2' -o -iname '*.png' -o -iname '*.jpg' \) 2>/dev/null | wc -l

/content/drive/MyDrive/colorferet
/content/drive/MyDrive/colorferet/colorferet
image files found:
50174


## 4. Sanity check on UCI Multiple Features (auto-downloads, no Drive needed)

In [10]:
!python experiments/run_mvda.py --dataset mfeat --mode concat --classifier ensemble

Loading dataset=mfeat scaler=robust ...
  views=6 dims=[76, 216, 64, 240, 47, 6] train=1000 test=1000 classes=10
Fitting MultiViewLDA(mode=concat, vc_lambda=0.0) ...
  shared-space dim = 9

=== mfeat | concat | ensemble ===
Accuracy: 98.700%

class   precision   recall      f1          support   
------------------------------------------------------
0       1.0000      0.9900      0.9950      100       
1       0.9709      1.0000      0.9852      100       
2       0.9900      0.9900      0.9900      100       
3       0.9796      0.9600      0.9697      100       
4       0.9901      1.0000      0.9950      100       
5       0.9898      0.9700      0.9798      100       
6       1.0000      0.9900      0.9950      100       
7       0.9804      1.0000      0.9901      100       
8       1.0000      0.9900      0.9950      100       
9       0.9703      0.9800      0.9751      100       
------------------------------------------------------
macro   0.9871      0.9870      0.9870    

## 5. Cross-pose face recognition on ColorFERET (MvDA)

Each pose is a view, each subject is a class. `run_feret.py` reduces each pose with PCA (eigenfaces), learns a shared MvDA subspace, and classifies held-out single-pose images by nearest class mean.

`--feret-poses` picks which poses become views; subjects missing **any** chosen pose are dropped, so more poses ⇒ fewer subjects. `fa`/`fb` are the most populated. The first run reads + caches images (slow); later runs reuse the cache.

In [11]:
!python experiments/run_feret.py \
    --feret-root "$FERET_ROOT" --feret-poses fa fb hl hr --feret-size 64 64 \
    --pca 120 --feret-max-subjects 200 --save feret_mvda.json

Loading ColorFERET from /content/drive/MyDrive/colorferet poses=['fa', 'fb', 'hl', 'hr'] ...
  views=4 samples/view=[882, 879, 825, 873] classes=200
Fitting MultiViewLDA(mode=mvda) on PCA-reduced views (dims=[120, 120, 120, 120]) ...
  shared-space dim = 199

Per-pose probe accuracy (NCM cosine):
pose    probes    accuracy    
------------------------------
fa      313       97.444      
fb      312       94.551      
hl      291       95.876      
hr      309       93.204      
------------------------------

=== ColorFERET | mvda | NCM(cosine) ===
Overall accuracy: 95.265%  (1225 probes, 200 subjects)
Macro F1: 0.9669


## 6. (optional) Frontal-only views (more subjects) and a component sweep

Using just the two frontal poses keeps the most subjects. Drop `--feret-max-subjects` to use everyone.

In [12]:
!python experiments/run_feret.py \
    --feret-root "$FERET_ROOT" --feret-poses fa fb --feret-size 64 64 --pca 120

Loading ColorFERET from /content/drive/MyDrive/colorferet poses=['fa', 'fb'] ...
  views=2 samples/view=[4089, 4074] classes=993
Fitting MultiViewLDA(mode=mvda) on PCA-reduced views (dims=[120, 120]) ...
  shared-space dim = 240

Per-pose probe accuracy (NCM cosine):
pose    probes    accuracy    
------------------------------
fa      1437      90.397      
fb      1432      90.922      
------------------------------

=== ColorFERET | mvda | NCM(cosine) ===
Overall accuracy: 90.659%  (2869 probes, 993 subjects)
Macro F1: 0.9383
